In [4]:
import torch
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import models
from tqdm import tqdm
import os
import sys

In [2]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, split, preprocess):
        self.split = split
        self.preprocess = preprocess

        with open(f'./data/{split}.csv', 'r') as f:
            lines = f.readlines()
        self.filename, self.y, self.a = [], [], []
        for line in lines[1:]:
            _, filename, y = line.rstrip().split(',')
            self.filename.append(filename)
            self.y.append(int(y))

    def __getitem__(self, sample_n):
        im = Image.open(f'./data/{self.split}/{self.filename[sample_n]}').convert('RGB')
        return self.preprocess(im), self.y[sample_n]
    
    def __len__(self):
        return len(self.y)

In [3]:
weights = models.ResNet50_Weights.IMAGENET1K_V1
resnet   = models.resnet50(weights=weights).eval()
resnet.fc = torch.nn.Identity()
preprocess = weights.transforms()
resnet.eval()
resnet.cuda()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
from torch import nn
p=0.5
model = nn.Sequential(
    nn.Dropout(p),
    nn.Linear(3, 1)
)
model.cuda()

In [ ]:
datasets = {split: Dataset(split, preprocess) for split in ['train', 'val']}
train_loader = DataLoader(datasets['train'], batch_size = 16, shuffle = True)
val_loader = DataLoader(datasets['val'], batch_size = 16, shuffle = False)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        pbar = tqdm(loader)

        for images, labels in pbar:
            images = images.cuda()
            labels = labels.float()

            if is_train:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            preds = (torch.sigmoid(logits) > 0.5).long()

            total_loss += loss.item() * images.size(0)
            total_correct += (preds == labels.long()).sum().item()
            total_samples += images.size(0)

            pbar.set_postfix(
                loss=total_loss / total_samples,
                acc=total_correct / total_samples,
            )

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )


for images, labels in tqdm(train_dl):
    images = images.cuda()
    
    with torch.no_grad():
        features = resnet(images)
        print(features.shape)
        break